In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
import sys
from mpi4py import MPI
import numpy as np

# torch
import torch

# quimb
import autoray as ar

from vmc_torch.experiment.tn_model import NNBF,NNBF_attention, NNBF_attention_Nsd, HFDS, init_weights_to_zero
from vmc_torch.hamiltonian_torch import spinful_Fermi_Hubbard_square_lattice_torch
from vmc_torch.fermion_utils import calc_phase_netket, from_quimb_config_to_netket_config, from_netket_config_to_quimb_config
from vmc_torch.sampler import MetropolisExchangeSamplerSpinful
from vmc_torch.variational_state import Variational_State
from vmc_torch.optimizer import SGD, SR, DecayScheduler
from vmc_torch.VMC import VMC
from vmc_torch.torch_utils import SVD,QR

from vmc_torch.utils import closest_divisible

# Register safe SVD and QR functions to torch
ar.register_function('torch','linalg.svd',SVD.apply)
ar.register_function('torch','linalg.qr',QR.apply)

COMM = MPI.COMM_WORLD
SIZE = COMM.Get_size()
RANK = COMM.Get_rank()

# Hamiltonian parameters
Lx = int(4)
Ly = int(4)
symmetry = 'Z2'
t = 1.0
U = 8.0
N_f = int(Lx*Ly-2)
n_fermions_per_spin = (N_f//2, N_f//2)

# Customized Hamiltonian elements to match with Ao's initial SD's definition
class spinful_Fermi_Hubbard_square_lattice_torch_Ao(spinful_Fermi_Hubbard_square_lattice_torch):
    def __init__(self, Lx, Ly, t, U, N_f, pbc=False, n_fermions_per_spin=None):
        super().__init__(Lx, Ly, t, U, N_f, pbc=pbc, n_fermions_per_spin=n_fermions_per_spin)

    def get_conn(self, sigma_quimb):
        """
        Return the connected configurations <eta| by the Hamiltonian to the state |sigma>,
        and their corresponding coefficients <eta|H|sigma>.
        """
        sigma = from_quimb_config_to_netket_config(sigma_quimb)
        connected_config_coeff = dict()
        for key, value in self._H.items():
            if len(key) == 3:
                # hopping term
                i0, j0, spin = key
                i = i0 if spin == 1 else i0 + self.hilbert.n_orbitals // 2
                j = j0 if spin == 1 else j0 + self.hilbert.n_orbitals // 2
                # Check if the two sites are different
                if sigma[i] != sigma[j]:
                    # H|sigma> = -t * |eta>
                    eta = sigma.copy()
                    eta[i], eta[j] = sigma[j], sigma[i]
                    eta_quimb0 = from_netket_config_to_quimb_config(eta)
                    eta_quimb = tuple(eta_quimb0)
                    # Calculate the phase correction
                    phase = calc_phase_netket(from_netket_config_to_quimb_config(sigma), eta_quimb0)
                    if eta_quimb not in connected_config_coeff:
                        connected_config_coeff[eta_quimb] = value*phase
                    else:
                        connected_config_coeff[eta_quimb] += value*phase
            elif len(key) == 1:
                # on-site term
                i = key[0]
                if sigma_quimb[i] == 3:
                    eta_quimb = tuple(sigma_quimb)
                    if eta_quimb not in connected_config_coeff:
                        connected_config_coeff[eta_quimb] = value
                    else:
                        connected_config_coeff[eta_quimb] += value
        
        return ar.do('array', list(connected_config_coeff.keys())), ar.do('array', list(connected_config_coeff.values()))
    
H = spinful_Fermi_Hubbard_square_lattice_torch_Ao(Lx, Ly, t, U, N_f, pbc=False, n_fermions_per_spin=n_fermions_per_spin)

graph = H.graph
# TN parameters
D = 4
chi = -1
dtype=torch.float64


# VMC sample size
N_samples = int(1e4)
N_samples = closest_divisible(N_samples, SIZE)
if (N_samples/SIZE)%2 != 0:
    N_samples += SIZE

# orbital_matrix_numpy = np.load(f"Hubbard4x4_{N_f}.npy")
# orbital_matrix = torch.tensor(orbital_matrix_numpy)
torch.manual_seed(42)
def kernel_init_from_matrix(tensor):
    # randomly initialize a tensor with the same shape as the input tensor
    tensor = torch.randn_like(tensor, dtype=dtype)
    return tensor

# model = NNBF(
#     hilbert=H.hilbert,
#     kernel_init=kernel_init_from_matrix,
#     param_dtype=dtype,
#     hidden_dim=32,
#     nn_eta=1.0,
# )
embed_dim=8
attention_heads=2
hidden_dim=embed_dim*Lx*Ly
# model = NNBF_attention(
#     nsite=Lx*Ly,
#     hilbert=H.hilbert,
#     # kernel_init=kernel_init_from_matrix,
#     param_dtype=dtype,
#     hidden_dim=hidden_dim,
#     embed_dim=embed_dim,
#     attention_heads=attention_heads,
#     nn_eta=1.0,
#     phys_dim=4,
# )
model = NNBF_attention_Nsd(
    nsite=Lx*Ly,
    hilbert=H.hilbert,
    kernel_init=kernel_init_from_matrix,
    param_dtype=dtype,
    hidden_dim=hidden_dim,
    embed_dim=embed_dim,
    attention_heads=attention_heads,
    attention_layers=1,
    position_wise_mlp_hidden_dim=embed_dim,
    nn_eta=1.0,
    phys_dim=4,
    Nsd=1
)

init_std = 1e-3
model.apply(lambda x: init_weights_to_zero(x, std=init_std))


model_names = {
    NNBF: 'NNBF',
    NNBF_attention: 'NNBF_attention',
    NNBF_attention_Nsd: f'NNBF_attention_Nsd={model.Nsd if hasattr(model, "Nsd") else ""}',
    HFDS: 'HFDS',
}
model_name = model_names.get(type(model), 'UnknownModel')

random_x = torch.tensor(H.hilbert.random_state(0))

model(random_x)

torch.Size([1, 16, 8])


tensor([-1015.8899], dtype=torch.float64, grad_fn=<StackBackward0>)

In [ ]:
model.backflow_transformers